<a href="https://colab.research.google.com/github/jaso01096/FoodClique---AI/blob/main/Copy_of_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import os
import sys
from collections import Counter

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import torch.nn as nn
import torch.optim as optim
import torchvision
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.notebook import tqdm

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

os.listdir("/content/drive/MyDrive/archive (2026)")

['EfficientNetB0-100-(224 X 224)- 98.40.h5',
 'sports.csv',
 'test',
 'train',
 'valid']

In [6]:
train_dir=os.path.join("/content/drive/MyDrive/archive (2026)","train")
class_directories=os.listdir(train_dir)
class_directories

['weightlifting',
 'wheelchair basketball',
 'wingsuit flying',
 'uneven bars',
 'volleyball',
 'water cycling',
 'ultimate',
 'water polo',
 'wheelchair racing',
 'tug of war',
 'track bicycle',
 'trapeze',
 'sumo wrestling',
 'tennis',
 'surfing',
 'skydiving',
 'table tennis',
 'steer wrestling',
 'swimming',
 'sky surfing',
 'sailboat racing',
 'speed skating',
 'shuffleboard',
 'shot put',
 'snowmobile racing',
 'snow boarding',
 'ski jumping',
 'sidecar racing',
 'rollerblade racing',
 'roller derby',
 'olympic wrestling',
 'rowing',
 'rugby',
 'rock climbing',
 'mushing',
 'parallel bar',
 'rings',
 'pole climbing',
 'pommel horse',
 'polo',
 'pole dancing',
 'nascar racing',
 'motorcycle racing',
 'pole vault',
 'judo',
 'luge',
 'lacrosse',
 'jai alai',
 'jousting',
 'log rolling',
 'ice climbing',
 'hydroplane racing',
 'horseshoe pitching',
 'javelin',
 'high jump',
 'ice yachting',
 'hurdles',
 'hockey',
 'horse racing',
 'horse jumping',
 'hammer throw',
 'giant slalom',
 

In [7]:
class ConvertToRGB:
  def __call__(self,img):
    if img.mode != "RGB":
      img=img.convert("RGB")
    return img

transform=transforms.Compose([
    ConvertToRGB(),
    transforms.Resize((224,224)),
    transforms.ToTensor() # Add this to convert PIL Images to PyTorch Tensors
])

In [8]:
os.listdir('/content/drive/MyDrive/archive (2026)/train/')

['weightlifting',
 'wheelchair basketball',
 'wingsuit flying',
 'uneven bars',
 'volleyball',
 'water cycling',
 'ultimate',
 'water polo',
 'wheelchair racing',
 'tug of war',
 'track bicycle',
 'trapeze',
 'sumo wrestling',
 'tennis',
 'surfing',
 'skydiving',
 'table tennis',
 'steer wrestling',
 'swimming',
 'sky surfing',
 'sailboat racing',
 'speed skating',
 'shuffleboard',
 'shot put',
 'snowmobile racing',
 'snow boarding',
 'ski jumping',
 'sidecar racing',
 'rollerblade racing',
 'roller derby',
 'olympic wrestling',
 'rowing',
 'rugby',
 'rock climbing',
 'mushing',
 'parallel bar',
 'rings',
 'pole climbing',
 'pommel horse',
 'polo',
 'pole dancing',
 'nascar racing',
 'motorcycle racing',
 'pole vault',
 'judo',
 'luge',
 'lacrosse',
 'jai alai',
 'jousting',
 'log rolling',
 'ice climbing',
 'hydroplane racing',
 'horseshoe pitching',
 'javelin',
 'high jump',
 'ice yachting',
 'hurdles',
 'hockey',
 'horse racing',
 'horse jumping',
 'hammer throw',
 'giant slalom',
 

In [9]:
dataset=datasets.ImageFolder(root=train_dir, transform=transform)

In [10]:
dataset_size = len(dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 10793
Validation dataset size: 2699


In [18]:
train_dataloader = DataLoader(train_dataset, batch_size=2)
val_dataloader = DataLoader(val_dataset, batch_size=2)

In [31]:
flatten=nn.Flatten()
model=nn.Sequential(
    nn.Flatten(),
    nn.Linear(3*224*224,512),
    nn.ReLU(),
    nn.Linear(512,100), # Changed output features to 100 to match the number of classes
    nn.ReLU(),
)

In [13]:
loss=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.01)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

Using device: cpu


In [15]:
def train(model, optimizer, loss_fn, data_loader, device="cpu"):
  model.train() # Set the model to training mode
  training_loss = 0.0
  for batch_idx, (inputs, targets) in enumerate(data_loader):
    optimizer.zero_grad()

    inputs = inputs.to(device)
    targets = targets.to(device)

    outputs = model(inputs)
    loss = loss_fn(outputs, targets)
    loss.backward()
    optimizer.step()

    training_loss += loss.item() * inputs.size(0)
     # Accumulate weighted loss

  return training_loss / len(data_loader.dataset) # Average loss per sample

train_loss_epoch = train(model, optimizer, loss, train_dataloader, device="cpu")
print(f"Training Loss: {train_loss_epoch:.4f}")

KeyboardInterrupt: 

In [22]:
# Modified train function to run for multiple epochs
def train_model(model, optimizer, loss_fn, train_loader, val_loader, num_epochs, device="cpu"):
    history = {'train_loss': [], 'val_loss': []}
    model.to(device)

    for epoch in range(num_epochs):
        model.train() # Set the model to training mode
        training_loss = 0.0
        train_samples = 0.0
        for batch_idx, (inputs, targets) in enumerate(train_loader):
          if batch_idx >= 5:
            break
          optimizer.zero_grad()

          inputs = inputs.to(device)
          targets = targets.to(device)

          outputs = model(inputs)
          loss = loss_fn(outputs, targets)
          loss.backward()
          optimizer.step()

          training_loss += loss.item() * inputs.size(0)
          train_samples += inputs.size(0)

        epoch_train_loss = training_loss / train_samples
        history['train_loss'].append(epoch_train_loss)

        # Validation phase
        model.eval() # Set the model to evaluation mode
        validation_loss = 0.0
        val_samples = 0.0
        with torch.no_grad():
            for batch_idx, (inputs, targets) in enumerate(val_loader):
              inputs = inputs.to(device)
              targets = targets.to(device)
              outputs = model(inputs)
              loss = loss_fn(outputs, targets)
              validation_loss += loss.item() * inputs.size(0)
              val_samples += inputs.size(0)

        epoch_val_loss = validation_loss / val_samples
        history['val_loss'].append(epoch_val_loss)

        print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}')

    return model, history

# Define the number of epochs
num_epochs = 1 # You can adjust this number

print(f"Starting training for {num_epochs} epochs...")
model, training_history = train_model(model, optimizer, loss, train_dataloader, val_dataloader, num_epochs, device=device)
print("Training complete!")

Starting training for 1 epochs...
Epoch 1/1 | Train Loss: 4.8287 | Val Loss: 5.3824
Training complete!


In [28]:
import torch.nn.functional as F

def predict(model,data_loader,device="cpu"):
  all_probs=torch.tensor([]).to(device)
  model.eval()
  with torch.no_grad():
    for batch_idx, (inputs,targets) in enumerate(data_loader):
      if batch_idx >= 10:
        break
      inputs = inputs.to(device)
      outputs = model(inputs)
      probs=F.softmax(outputs,dim=1) # Fixed: 'output' changed to 'outputs'
      all_probs=torch.cat((all_probs,probs),dim=0)
  return all_probs

In [29]:
# Re-run prediction with the trained model on the correct device
predicted_probs = predict(model, val_dataloader, device=device)

# Get the maximum probability and the predicted class index for each sample
max_probs, predicted_classes = torch.max(predicted_probs, dim=1)

print("Predicted Probabilities (first 5 samples) after training:\n", predicted_probs[:5])
print("Maximum Probabilities (first 5 samples) after training:\n", max_probs[:5])
print("Predicted Classes (first 5 samples) after training:\n", predicted_classes[:5])

Predicted Probabilities (first 5 samples) after training:
 tensor([[0.0046, 0.0050, 0.0046,  ..., 0.0046, 0.0046, 0.0046],
        [0.0046, 0.0050, 0.0046,  ..., 0.0046, 0.0046, 0.0046],
        [0.0046, 0.0050, 0.0046,  ..., 0.0046, 0.0046, 0.0046],
        [0.0046, 0.0050, 0.0046,  ..., 0.0046, 0.0046, 0.0046],
        [0.0046, 0.0050, 0.0046,  ..., 0.0046, 0.0046, 0.0046]])
Maximum Probabilities (first 5 samples) after training:
 tensor([0.0050, 0.0050, 0.0050, 0.0050, 0.0050])
Predicted Classes (first 5 samples) after training:
 tensor([69, 69, 69, 69, 69])


In [30]:
num_classes = len(dataset.classes)
print(f"Number of unique classes in the dataset: {num_classes}")
print(f"First 10 class names: {dataset.classes[:10]}")

Number of unique classes in the dataset: 100
First 10 class names: ['air hockey', 'ampute football', 'archery', 'arm wrestling', 'axe throwing', 'balance beam', 'barell racing', 'baseball', 'basketball', 'baton twirling']
